# Практика 05. Метод опорных векторов

**Версия:** 2026-09-25 (f95bbe8)

**Как сдавать работу**

1. Откройте ноутбук в Colab и сохраните копию себе: «Файл → Сохранить копию на Диске». Работайте в копии.
2. Выполните задания: код пишите вместо `# ВАШ КОД ЗДЕСЬ` / `raise NotImplementedError`,
   ответы на вопросы — после «*Ваш ответ:*».
3. После каждого задания запускайте ячейку с проверками. Проверки — для самоконтроля:
   их прохождение не гарантирует зачёт, а текстовые ответы проверяются отдельно.
4. Перед сдачей выполните «Среда выполнения → Перезапустить сеанс и выполнить все»: ноутбук должен
   выполниться целиком без ошибок.
5. Откройте доступ по ссылке («Настройки доступа → Все, у кого есть ссылка») и вставьте ссылку
   на свою копию в таблицу курса.

Свёрнутые ячейки со значком ▶ — служебные (загрузка данных, функции проверки). Их нужно выполнять,
но менять не нужно.

К лекции 05. План работы:

1. **Часть 1** — ядро SVM своими руками на NumPy: RBF-ядро, решающая функция по двойственным переменным,
   проверка условий ККТ на обученной модели, линейный SVM субградиентным спуском. Каждый результат
   сверяется со scikit-learn.
2. **Часть 2** — SVM из scikit-learn на данных Spambase (распознавание спама): влияние масштабирования,
   линейное и RBF-ядро, подбор $C$ и $\gamma$ кросс-валидацией, выбор порога для спам-фильтра.
3. **Часть 3** — эксперимент и выводы: границы решений разных ядер на нелинейно разделимых данных, влияние
   $C$ и $\gamma$ на границу, число опорных векторов и качество. Код здесь простой, оценивается объяснение.

In [ ]:
# @title Служебная ячейка: импорты и функции проверки { display-mode: "form" }
import inspect
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 0
rng = np.random.default_rng(SEED)


def assert_no_loops(func):
    """Проверяет, что в теле функции нет циклов for/while (включения списков тоже считаются)."""
    source = inspect.getsource(func)
    body = source.split('"""')[-1] if '"""' in source else source
    assert not re.search(r"\b(for|while)\b", body), (
        f"В функции {func.__name__} есть цикл. Здесь нужно решение без циклов — операциями над массивами."
    )


def plot_svm(model, X, y, ax, title="", margin=True):
    """Рисует объекты двух классов (метки -1 и +1), области классов и границу SVM.

    Сплошная линия — граница f(x) = 0, пунктир — края полосы f(x) = ±1, обведены опорные векторы.
    """
    pad = 0.3
    g0, g1 = np.meshgrid(
        np.linspace(X[:, 0].min() - pad, X[:, 0].max() + pad, 250),
        np.linspace(X[:, 1].min() - pad, X[:, 1].max() + pad, 250),
    )
    f = model.decision_function(np.c_[g0.ravel(), g1.ravel()]).reshape(g0.shape)
    ax.contourf(g0, g1, f > 0, levels=[-0.5, 0.5, 1.5], colors=["#deebf7", "#fde0dd"])
    levels, styles = ([-1, 0, 1], ["--", "-", "--"]) if margin else ([0], ["-"])
    ax.contour(g0, g1, f, levels=levels, colors="k", linestyles=styles, linewidths=0.8)
    ax.scatter(*X[y == -1].T, s=12, color="#1f77b4", edgecolor="k", linewidth=0.3)
    ax.scatter(*X[y == 1].T, s=12, color="#d62728", marker="s", edgecolor="k", linewidth=0.3)
    if hasattr(model, "support_vectors_"):
        ax.scatter(*model.support_vectors_.T, s=45, facecolor="none", edgecolor="k", linewidth=0.6)
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])


print("Готово")

# Часть 1. SVM своими руками

Напомним основные формулы лекции. Метки классов — $y_i \in \{-1, +1\}$. SVM с мягким запасом и ядром $K$
решает двойственную задачу
$$
\max_{\boldsymbol{\alpha}} \sum_{i=1}^n \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j K(\mathbf{x}_i, \mathbf{x}_j)
\quad \text{при} \quad \sum_{i=1}^n \alpha_i y_i = 0, \quad 0 \leq \alpha_i \leq C,
$$
а классифицирует объект знаком решающей функции
$$
f(\mathbf{x}) = \sum_{i=1}^n \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b,
$$
где в сумме участвуют только опорные векторы — объекты с $\alpha_i > 0$.

## Задание 1.1. RBF-ядро

Напишите `rbf_kernel(X, Z, gamma)`, возвращающую матрицу $\bigl(K(\mathbf{x}_i, \mathbf{z}_j)\bigr)_{i,j}$ размера
`len(X) × len(Z)`, где
$$
K(\mathbf{x}, \mathbf{z}) = \exp\bigl(-\gamma\left\lVert \mathbf{x} - \mathbf{z} \right\rVert^2\bigr).
$$
Без циклов. Квадраты расстояний для всех пар сразу удобно считать по формуле
$\left\lVert \mathbf{x} - \mathbf{z} \right\rVert^2 = \left\lVert \mathbf{x} \right\rVert^2 + \left\lVert \mathbf{z} \right\rVert^2 - 2\mathbf{x}^{\top}\mathbf{z}$. Из-за ошибок округления результат может
получиться чуть меньше нуля — обрежьте его снизу нулём (`np.maximum`).

In [ ]:
def rbf_kernel(X, Z, gamma):
    """Матрица RBF-ядра между строками X (n × d) и строками Z (m × d)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel as sk_rbf_kernel

r = np.random.default_rng(1)
X_chk, Z_chk = r.normal(size=(6, 3)), r.normal(size=(4, 3))
K_chk = rbf_kernel(X_chk, Z_chk, gamma=0.5)
assert K_chk.shape == (6, 4), f"Матрица ядра должна иметь размер 6 × 4, получено {np.shape(K_chk)}"
assert np.allclose(K_chk, sk_rbf_kernel(X_chk, Z_chk, gamma=0.5)), "Значения ядра не совпали со sklearn.metrics.pairwise.rbf_kernel"
assert np.allclose(rbf_kernel(X_chk, X_chk, gamma=2.0), sk_rbf_kernel(X_chk, X_chk, gamma=2.0)), "Неверный результат при gamma = 2: параметр gamma должен умножать квадрат расстояния"
assert np.allclose(np.diag(rbf_kernel(X_chk, X_chk, 1.0)), 1), "K(x, x) должно быть равно 1"
X_big = r.normal(size=(50, 2)) * 1000
assert np.all(rbf_kernel(X_big, X_big, 1e-9) <= 1), "Значения ядра не могут быть больше 1: обрежьте квадрат расстояния снизу нулём"
assert_no_loops(rbf_kernel)
print("OK")

## Задание 1.2. Решающая функция по двойственным переменным

Обученный `SVC` хранит всё, что нужно для формулы $f(\mathbf{x}) = \sum_i \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b$:

- `support_vectors_` — опорные векторы $\mathbf{x}_i$ (матрица $n_{\text{SV}} \times d$);
- `dual_coef_` — произведения $\alpha_i y_i$ для опорных векторов, матрица размера $1 \times n_{\text{SV}}$;
- `intercept_` — смещение $b$ (массив из одного числа).

Напишите `svm_decision_function(X, support_vectors, dual_coef, intercept, gamma)`: значения $f(\mathbf{x})$ для всех
строк `X`, вектор длины `len(X)`. Используйте свою `rbf_kernel`. Без циклов.

In [ ]:
def svm_decision_function(X, support_vectors, dual_coef, intercept, gamma):
    """f(x) = sum_i alpha_i y_i K(x_i, x) + b для всех строк X."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.datasets import make_moons
from sklearn.svm import SVC

X_moons, y01 = make_moons(200, noise=0.2, random_state=SEED)
y_moons = 2 * y01 - 1  # метки {0, 1} -> {-1, +1}
svc_moons = SVC(kernel="rbf", gamma=1.0, C=1.0, tol=1e-6).fit(X_moons, y_moons)

X_new = np.random.default_rng(2).uniform(-1.5, 2.5, size=(300, 2))
f_ours = svm_decision_function(
    X_new, svc_moons.support_vectors_, svc_moons.dual_coef_, svc_moons.intercept_, gamma=1.0
)
f_sk = svc_moons.decision_function(X_new)
assert np.shape(f_ours) == (300,), f"Должен получиться вектор длины 300, получено {np.shape(f_ours)}"
assert np.allclose(f_ours, f_sk), (
    f"Значения f(x) расходятся с decision_function (макс. разница {np.abs(f_ours - f_sk).max():.4f}). "
    "Проверьте знак коэффициентов, смещение и порядок аргументов ядра"
)
assert np.array_equal(np.sign(f_ours), svc_moons.predict(X_new)), "Знак f(x) должен совпадать с predict"
assert_no_loops(svm_decision_function)
print("OK")

## Задание 1.3. Условия ККТ и типы объектов

По условиям ККТ множитель $\alpha_i$ говорит, где лежит объект относительно полосы:

- $\alpha_i = 0$ — объект не опорный, его отступ $M_i = y_i f(\mathbf{x}_i) \geq 1$;
- $0 < \alpha_i < C$ — опорный вектор на краю полосы, $M_i = 1$;
- $\alpha_i = C$ — опорный вектор-нарушитель, $M_i \leq 1$.

Проверим это на модели `svc_moons` ($C = 1$). Сначала соберите:

- `alpha` — вектор $\alpha_i$ для **всех** 200 объектов выборки: нули для неопорных, а для опорных — модули
  `dual_coef_` (так как $\alpha_i \geq 0$, а $y_i = \pm 1$). Номера опорных векторов в выборке — `svc_moons.support_`;
- `margins` — отступы $M_i = y_i f(\mathbf{x}_i)$ всех объектов (используйте `svc_moons.decision_function`).

Затем напишите `sv_groups(alpha, C, eps=1e-8)`, возвращающую три булевых маски `(non_sv, free_sv, bound_sv)`:
$\alpha_i \leq \varepsilon$; $\varepsilon < \alpha_i < C - \varepsilon$; $\alpha_i \geq C - \varepsilon$. Сравнивать с допуском $\varepsilon$
приходится потому, что $\alpha$ найдены численно. Без циклов.

In [ ]:
C_MOONS = 1.0
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError


def sv_groups(alpha, C, eps=1e-8):
    """Маски (неопорные, опорные на краю полосы, опорные-нарушители)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


non_sv, free_sv, bound_sv = sv_groups(alpha, C_MOONS)
print(f"неопорных: {non_sv.sum()}, на краю полосы: {free_sv.sum()}, нарушителей: {bound_sv.sum()}")

In [ ]:
assert np.shape(alpha) == (200,) and np.shape(margins) == (200,), "alpha и margins — векторы длины 200, по числу объектов"
assert np.all(alpha >= 0) and np.all(alpha <= C_MOONS + 1e-9), "Множители должны лежать в [0, C]: возьмите модули dual_coef_"
assert np.count_nonzero(alpha) == len(svc_moons.support_), "Ненулевые alpha должны быть ровно у опорных векторов (svc_moons.support_)"
assert np.isclose(np.sum(alpha * y_moons), 0), "Должно выполняться условие sum(alpha_i * y_i) = 0 — проверьте, в каком порядке расставлены alpha"
assert np.allclose(margins, y_moons * svc_moons.decision_function(X_moons)), "margins — это y_i * f(x_i)"
masks = sv_groups(alpha, C_MOONS)
assert all(m.dtype == bool and m.shape == (200,) for m in masks), "sv_groups должна вернуть три булевых маски длины 200"
assert np.all(masks[0].astype(int) + masks[1] + masks[2] == 1), "Каждый объект должен попасть ровно в одну группу"
assert masks[1].sum() > 0 and masks[2].sum() > 0, "В этой задаче есть и опорные векторы на краю полосы, и нарушители"
assert np.all(margins[non_sv] >= 1 - 1e-4), "ККТ: у неопорных объектов отступ должен быть не меньше 1"
assert np.allclose(margins[free_sv], 1, atol=1e-4), "ККТ: у опорных векторов с 0 < alpha < C отступ должен быть равен 1"
assert np.all(margins[bound_sv] <= 1 + 1e-4), "ККТ: у опорных векторов с alpha = C отступ не больше 1"
assert np.array_equal(sv_groups(np.array([0.0, 0.5, 1.0, 1e-12]), 1.0)[1], [False, True, False, False]), "Проверьте границы групп на простом примере"
assert_no_loops(sv_groups)
print("OK")

## Задание 1.4. Линейный SVM субградиентным спуском

Прямая задача SVM с мягким запасом эквивалентна минимизации функционала
$$
Q(\mathbf{w}, b) = \frac{\tau}{2}\left\lVert \mathbf{w} \right\rVert^2 + \frac{1}{n}\sum_{i=1}^n \max\bigl(0,\ 1 - y_i(\mathbf{w}^{\top}\mathbf{x}_i + b)\bigr),
\qquad \tau = \frac{1}{nC}.
$$
Его субградиент:
$$
\nabla_{\mathbf{w}} Q = \tau\mathbf{w} - \frac{1}{n}\sum_{i=1}^n \mathbb{I}\left[y_i f(\mathbf{x}_i) < 1\right]\, y_i \mathbf{x}_i, \qquad
\frac{\partial Q}{\partial b} = -\frac{1}{n}\sum_{i=1}^n \mathbb{I}\left[y_i f(\mathbf{x}_i) < 1\right]\, y_i.
$$
Напишите:

- `svm_objective(w, b, X, y, tau)` — значение $Q$ (число);
- `svm_subgradient(w, b, X, y, tau)` — пару `(grad_w, grad_b)`; обе функции без циклов;
- `train_svm_gd(X, y, tau, lr, n_iter)` — субградиентный спуск с постоянным шагом `lr` из $\mathbf{w} = 0$, $b = 0$:
  `n_iter` шагов $\mathbf{w} \leftarrow \mathbf{w} - \eta\nabla_{\mathbf{w}}Q$, $b \leftarrow b - \eta\,\partial Q/\partial b$. Функция возвращает
  `(w, b, history)`, где `history` — список значений $Q$ **после** каждого шага.

In [ ]:
def svm_objective(w, b, X, y, tau):
    """Функционал Q(w, b): L2-регуляризатор плюс средняя hinge loss."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def svm_subgradient(w, b, X, y, tau):
    """Субградиент Q по w и по b."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.datasets import make_blobs

X_lin, y01 = make_blobs(300, centers=[[-1, -1], [1.2, 1]], cluster_std=1.1, random_state=1)
y_lin = 2 * y01 - 1
TAU = 0.01

w0, b0 = np.array([0.5, -0.3]), 0.2
margins0 = y_lin * (X_lin @ w0 + b0)
q0 = TAU / 2 * 0.34 + np.mean(np.maximum(0, 1 - margins0))
assert np.isclose(svm_objective(w0, b0, X_lin, y_lin, TAU), q0), "svm_objective: значение Q посчитано неверно"
assert np.isclose(svm_objective(np.zeros(2), 0.0, X_lin, y_lin, TAU), 1.0), "При w = 0, b = 0 все отступы нулевые и Q = 1"

gw, gb = svm_subgradient(w0, b0, X_lin, y_lin, TAU)
assert np.shape(gw) == (2,) and np.ndim(gb) == 0, "grad_w — вектор длины d, grad_b — число"
h = 1e-6
num_gw = np.array([
    (svm_objective(w0 + h * e, b0, X_lin, y_lin, TAU) - svm_objective(w0 - h * e, b0, X_lin, y_lin, TAU)) / (2 * h)
    for e in np.eye(2)
])
num_gb = (svm_objective(w0, b0 + h, X_lin, y_lin, TAU) - svm_objective(w0, b0 - h, X_lin, y_lin, TAU)) / (2 * h)
assert np.allclose(gw, num_gw, atol=1e-5), f"grad_w не совпадает с численной производной: {gw} против {num_gw}"
assert np.isclose(gb, num_gb, atol=1e-5), f"grad_b не совпадает с численной производной: {gb} против {num_gb}"
assert_no_loops(svm_objective)
assert_no_loops(svm_subgradient)
print("OK")

Теперь сам спуск.

In [ ]:
def train_svm_gd(X, y, tau, lr, n_iter):
    """Субградиентный спуск с постоянным шагом. Возвращает (w, b, history)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
w_gd, b_gd, history = train_svm_gd(X_lin, y_lin, TAU, lr=0.1, n_iter=2000)

# Эталон: та же задача, решённая SVC через двойственную задачу, C = 1 / (n * tau).
svc_lin = SVC(kernel="linear", C=1 / (len(y_lin) * TAU), tol=1e-6).fit(X_lin, y_lin)
w_sk, b_sk = svc_lin.coef_[0], svc_lin.intercept_[0]
q_gd, q_sk = svm_objective(w_gd, b_gd, X_lin, y_lin, TAU), svm_objective(w_sk, b_sk, X_lin, y_lin, TAU)
print(f"GD:  w = {np.round(w_gd, 3)}, b = {b_gd:.3f}, Q = {q_gd:.5f}")
print(f"SVC: w = {np.round(w_sk, 3)}, b = {b_sk:.3f}, Q = {q_sk:.5f}")

plt.figure(figsize=(6, 3))
plt.plot(history)
plt.axhline(q_sk, color="k", ls="--", lw=0.8, label="оптимум (SVC)")
plt.yscale("log")
plt.xlabel("итерация")
plt.ylabel("Q(w, b)")
plt.legend()
plt.show()

assert len(history) == 2000, f"history должна содержать 2000 значений, получено {len(history)}"
assert np.isclose(history[-1], q_gd), "Последний элемент history — значение Q в итоговой точке"
assert history[0] < 1.0, "Уже после первого шага Q должна стать меньше 1 (значения в точке w = 0)"
assert q_gd - q_sk < 1e-4, f"Спуск не дошёл до минимума: Q = {q_gd:.5f}, а минимум {q_sk:.5f}"
assert np.allclose(w_gd, w_sk, atol=0.01) and abs(b_gd - b_sk) < 0.01, "Веса должны совпасть с решением SVC с точностью 0.01"
print("OK")

# Часть 2. SVM на данных Spambase

Данные Spambase (UCI, 4601 электронное письмо, собранные в Hewlett-Packard в 1999 году). Нужно определить,
спам ли письмо (класс 1, около 39% писем). Признаки посчитаны по тексту письма:

- 48 признаков `word_freq_*` — доля слов письма (в процентах), совпадающих с данным словом: `free`, `money`,
  `remove`, `you`, `hp`, `george` и т. п.;
- 6 признаков `char_freq_*` — доля символов `;`, `(`, `[`, `!`, `$`, `#`;
- 3 признака `capital_run_length_*` — средняя и наибольшая длина последовательностей заглавных букв и общее
  число заглавных букв.

In [ ]:
# @title Загрузка данных: Spambase (OpenML, id 44) и разбиение { display-mode: "form" }
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

spam = fetch_openml(data_id=44, as_frame=True, parser="auto").frame
y_all = spam["class"].astype(int).to_numpy()
X_all = spam.drop(columns=["class"])
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all
)
print(f"обучение: {X_train.shape}, тест: {X_test.shape}, доля спама: {y_all.mean():.3f}")
X_train.describe().T[["mean", "std", "max"]].round(2).tail(8)

## Задание 2.1. Масштабирование

Сравните SVM с RBF-ядром с параметрами по умолчанию (`SVC()`) без масштабирования признаков и с ним.
Посчитайте долю верных ответов на кросс-валидации `CV` по обучающей выборке (`cross_val_score`):

- `cv_raw` — средняя accuracy для `SVC()` на исходных признаках;
- `cv_scaled` — средняя accuracy для конвейера `Pipeline([("scaler", StandardScaler()), ("svc", SVC())])`.

Масштабирование стоит внутри конвейера: тогда на каждом шаге кросс-валидации среднее и стандартное
отклонение признаков оцениваются только по обучающим блокам.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

CV = StratifiedKFold(5, shuffle=True, random_state=SEED)

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print(f"без масштабирования: {cv_raw:.3f}, со стандартизацией: {cv_scaled:.3f}")

In [ ]:
assert 0.6 < cv_raw < 0.8, f"cv_raw = {cv_raw:.3f}: ожидалась accuracy SVC() на исходных признаках (около 0.7)"
assert cv_scaled > 0.92, f"cv_scaled = {cv_scaled:.3f}: со стандартизацией accuracy должна быть выше 0.92"
print("OK")

## Задание 2.2. Линейное ядро

Подберите $C$ для SVM с линейным ядром. Конвейер — `StandardScaler` и `SVC(kernel="linear")` с именами шагов
`"scaler"` и `"svc"`, сетка по $C$ — `C_LINEAR`, кросс-валидация — `CV`, метрика — accuracy (по умолчанию).
Результат — `grid_linear`, обученный на `X_train`.

In [ ]:
from sklearn.model_selection import GridSearchCV

C_LINEAR = [0.001, 0.01, 0.1, 1, 10]

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("лучшее C:", grid_linear.best_params_, f"accuracy (CV) = {grid_linear.best_score_:.3f}")

In [ ]:
assert isinstance(grid_linear, GridSearchCV), "grid_linear должен быть GridSearchCV"
assert grid_linear.best_estimator_.named_steps["svc"].kernel == "linear", "Нужно линейное ядро: SVC(kernel='linear')"
assert "scaler" in grid_linear.best_estimator_.named_steps, "Первый шаг конвейера — StandardScaler с именем 'scaler'"
assert len(grid_linear.cv_results_["params"]) == len(C_LINEAR), "В сетке должны быть все значения C_LINEAR"
assert grid_linear.cv.random_state == SEED and grid_linear.cv.get_n_splits() == 5, "Используйте кросс-валидацию CV"
assert grid_linear.best_score_ > 0.92, f"accuracy на кросс-валидации подозрительно низкая: {grid_linear.best_score_:.3f}"
print("OK")

## Задание 2.3. RBF-ядро: подбор $C$ и $\gamma$

Подберите $C$ и $\gamma$ для SVM с RBF-ядром поиском по сетке `C_GRID` × `GAMMA_GRID`. Конвейер — как
в задании 2.2, но с `SVC(kernel="rbf")`. Укажите `return_train_score=True`: качество на обучающих блоках
понадобится в части 3. Результат — `grid_rbf`. Обучение займёт около минуты.

In [ ]:
C_GRID = [0.1, 1, 10, 100, 1000]
GAMMA_GRID = [1e-4, 1e-3, 1e-2, 1e-1, 1]

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("лучшие параметры:", grid_rbf.best_params_, f"accuracy (CV) = {grid_rbf.best_score_:.3f}")

In [ ]:
assert isinstance(grid_rbf, GridSearchCV), "grid_rbf должен быть GridSearchCV"
assert grid_rbf.best_estimator_.named_steps["svc"].kernel == "rbf", "Нужно RBF-ядро"
assert "scaler" in grid_rbf.best_estimator_.named_steps, "Первый шаг конвейера — StandardScaler с именем 'scaler'"
assert len(grid_rbf.cv_results_["params"]) == len(C_GRID) * len(GAMMA_GRID), "В сетке должны быть все пары (C, gamma)"
assert "mean_train_score" in grid_rbf.cv_results_, "Нужен return_train_score=True"
assert grid_rbf.cv.random_state == SEED and grid_rbf.cv.get_n_splits() == 5, "Используйте кросс-валидацию CV"
assert grid_rbf.best_score_ > 0.93, f"accuracy на кросс-валидации подозрительно низкая: {grid_rbf.best_score_:.3f}"
print("OK")

## Задание 2.4. Качество на тестовой выборке

Заполните словарь `results`: для лучших моделей `"linear"` и `"rbf"` (`best_estimator_` из заданий 2.2 и 2.3) —
словари с accuracy, precision и recall на тестовой выборке. Положительный класс — спам (1). Добавьте ещё
`"n_sv"` — число опорных векторов модели (`model.named_steps["svc"].n_support_` — по классам, нужна сумма).

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

models = {"linear": grid_linear.best_estimator_, "rbf": grid_rbf.best_estimator_}
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

pd.DataFrame(results).T.round(4)

In [ ]:
assert set(results) == {"linear", "rbf"}, f"Нужны ключи linear и rbf, получено {set(results)}"
for name, model in models.items():
    pred = model.predict(X_test)
    assert np.isclose(results[name]["accuracy"], accuracy_score(y_test, pred)), f"{name}: accuracy посчитана неверно — нужны предсказания на X_test"
    assert np.isclose(results[name]["precision"], precision_score(y_test, pred)), f"{name}: precision посчитана неверно — положительный класс 1 (спам)"
    assert np.isclose(results[name]["recall"], recall_score(y_test, pred)), f"{name}: recall посчитан неверно — положительный класс 1 (спам)"
    assert results[name]["n_sv"] == model.named_steps["svc"].n_support_.sum(), f"{name}: n_sv — сумма n_support_ по классам"
print("OK")

## Задание 2.5. Порог для спам-фильтра

Ошибки спам-фильтра неравноценны: пропущенный спам раздражает, а нужное письмо, попавшее в спам, может
обойтись дорого. Потребуем, чтобы precision (доля настоящего спама среди отправленного в спам) была не
ниже 99%, а recall при этом — как можно выше.

SVM не выдаёт вероятностей, но значение решающей функции $f(\mathbf{x})$ (`decision_function`) — мера уверенности,
и порог можно сдвинуть: объявлять спамом письма с $f(\mathbf{x}) \geq t$. Выбирать порог по тестовой выборке нельзя —
тогда оценка качества на ней станет завышенной. Поэтому:

1. получите значения $f(\mathbf{x})$ для обучающей выборки «честно», по кросс-валидации: `cross_val_predict` с
   лучшей RBF-моделью `grid_rbf.best_estimator_`, `cv=CV`, `method="decision_function"` → `scores_cv`;
2. по ним найдите `threshold` — **наименьший** порог, при котором precision на обучающей выборке не ниже 0.99.
   Удобно использовать `precision_recall_curve(y_train, scores_cv)`: она возвращает массивы precision, recall
   и порогов; `precision[k]` соответствует порогу `thresholds[k]` (у `precision` на один элемент больше);
3. примените порог к тестовой выборке: `pred_thr = (f(x) >= threshold)` для модели `grid_rbf.best_estimator_`,
   посчитайте `precision_thr`, `recall_thr` и `n_lost` — число нормальных писем (класс 0), попавших в спам.

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import cross_val_predict

best_rbf = grid_rbf.best_estimator_
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

pred_0 = best_rbf.predict(X_test)
print(f"порог 0:     precision = {precision_score(y_test, pred_0):.3f}, recall = {recall_score(y_test, pred_0):.3f}, "
      f"потеряно писем: {np.sum((pred_0 == 1) & (y_test == 0))}")
print(f"порог {threshold:.2f}: precision = {precision_thr:.3f}, recall = {recall_thr:.3f}, потеряно писем: {n_lost}")

In [ ]:
assert np.shape(scores_cv) == (len(y_train),), "scores_cv — значения f(x) для всех объектов обучающей выборки"
assert not np.allclose(scores_cv, best_rbf.decision_function(X_train)), (
    "scores_cv должны быть получены по кросс-валидации (cross_val_predict), а не моделью, обученной на всей X_train"
)
assert precision_score(y_train, scores_cv >= threshold) >= 0.99, "При выбранном пороге precision на кросс-валидации должна быть не ниже 0.99"
lower = scores_cv[scores_cv < threshold]
assert len(lower) == 0 or precision_score(y_train, scores_cv >= lower.max()) < 0.99, (
    "Порог должен быть наименьшим из подходящих: следующий меньший порог уже не даёт precision 0.99"
)
assert threshold > 0, "Порог для высокой precision должен быть больше 0"
test_scores = best_rbf.decision_function(X_test)
assert np.isclose(precision_thr, precision_score(y_test, test_scores >= threshold)), "precision_thr посчитана неверно"
assert np.isclose(recall_thr, recall_score(y_test, test_scores >= threshold)), "recall_thr посчитан неверно"
assert n_lost == np.sum((test_scores >= threshold) & (y_test == 0)), "n_lost — число писем класса 0, объявленных спамом"
print("OK")

# Часть 3. Эксперимент и выводы

## Задание 3.1. Ядра на нелинейно разделимых данных

Возьмём два синтетических набора, которые нельзя разделить прямой: «концентрические окружности» и
«две луны». Для каждого из них обучите на обучающей половине три SVM ($C = 1$):

- `"linear"` — `SVC(kernel="linear")`;
- `"poly"` — `SVC(kernel="poly", degree=2, coef0=1)` для окружностей и `degree=3` для лун;
- `"rbf"` — `SVC(kernel="rbf", gamma=1)`.

Сохраните долю верных ответов на тестовой половине в словарь `kernel_scores`: ключ — пара
`(имя_набора, ядро)`, например `("circles", "rbf")`. Нарисуйте границы всех шести моделей на сетке графиков
2 × 3 функцией `plot_svm` (в заголовке — ядро и accuracy на тесте).

In [ ]:
from sklearn.datasets import make_circles

datasets = {}
for name, (X_d, y_d) in {
    "circles": make_circles(400, noise=0.15, factor=0.5, random_state=SEED),
    "moons": make_moons(400, noise=0.25, random_state=SEED),
}.items():
    datasets[name] = train_test_split(X_d, 2 * y_d - 1, test_size=0.5, random_state=SEED, stratify=y_d)

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert len(kernel_scores) == 6, "Нужно 6 значений: 2 набора × 3 ядра"
assert kernel_scores[("circles", "linear")] < 0.7, "Линейное ядро не должно справляться с окружностями — проверьте, что модель обучена на circles"
assert kernel_scores[("circles", "poly")] > 0.9 and kernel_scores[("circles", "rbf")] > 0.9, "Полиномиальное и RBF-ядро должны разделять окружности"
assert kernel_scores[("moons", "rbf")] > kernel_scores[("moons", "linear")], "На лунах RBF должно быть лучше линейного ядра"
print("OK")

## Задание 3.2. Влияние $C$ и $\gamma$

На «двух лунах» (`datasets["moons"]`) обучите `SVC(kernel="rbf")` для всех пар $C$ из `C_MOONS_GRID` и $\gamma$ из
`GAMMA_MOONS_GRID`. Для каждой пары сохраните в словари с ключом `(C, gamma)`:

- `n_sv` — число опорных векторов (`len(model.support_)`);
- `train_acc`, `test_acc` — accuracy на обучающей и тестовой половинах.

Нарисуйте границы на сетке 3 × 3 (строки — $C$, столбцы — $\gamma$) с `plot_svm(..., margin=False)`; в заголовке —
$C$, $\gamma$, число опорных векторов и обе accuracy.

In [ ]:
C_MOONS_GRID = [0.1, 1, 100]
GAMMA_MOONS_GRID = [0.1, 1, 30]
X_tr, X_te, y_tr, y_te = datasets["moons"]

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
keys = {(C, g) for C in C_MOONS_GRID for g in GAMMA_MOONS_GRID}
assert set(n_sv) == keys and set(train_acc) == keys and set(test_acc) == keys, "Нужны значения для всех 9 пар (C, gamma)"
assert n_sv[(0.1, 1)] > n_sv[(1, 1)] > n_sv[(100, 1)], "При фиксированном gamma число опорных векторов должно убывать с ростом C"
assert train_acc[(100, 30)] > test_acc[(100, 30)] + 0.1, "При больших C и gamma модель должна переобучаться: train заметно выше test"
assert all(0 <= v <= 1 for v in list(train_acc.values()) + list(test_acc.values())), "Accuracy — доля, от 0 до 1"
print("OK")

## Задание 3.3. Карта качества по сетке $C \times \gamma$

По `grid_rbf.cv_results_` из задания 2.3 постройте две тепловые карты — accuracy на обучающих блоках
(`mean_train_score`) и на кросс-валидации (`mean_test_score`). Сохраните их в матрицы `train_map` и `cv_map`
размера `len(C_GRID) × len(GAMMA_GRID)`: строка — $C$, столбец — $\gamma$. Порядок параметров в `cv_results_["params"]`
проверьте сами. Для рисования подойдёт `plt.imshow` с подписями осей и значениями в клетках.

In [ ]:
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert np.shape(train_map) == (len(C_GRID), len(GAMMA_GRID)) and np.shape(cv_map) == (len(C_GRID), len(GAMMA_GRID)), (
    "Матрицы должны иметь размер len(C_GRID) × len(GAMMA_GRID)"
)
for k, params in enumerate(grid_rbf.cv_results_["params"]):
    i, j = C_GRID.index(params["svc__C"]), GAMMA_GRID.index(params["svc__gamma"])
    assert np.isclose(cv_map[i, j], grid_rbf.cv_results_["mean_test_score"][k]), (
        f"cv_map[{i}, {j}] не соответствует C = {params['svc__C']}, gamma = {params['svc__gamma']}: строка — C, столбец — gamma"
    )
    assert np.isclose(train_map[i, j], grid_rbf.cv_results_["mean_train_score"][k]), "train_map заполнена неверно"
assert np.isclose(cv_map.max(), grid_rbf.best_score_), "Максимум cv_map должен совпасть с best_score_"
print("OK")

## Задание 3.4. Выводы

Ответьте на вопросы, опираясь на свои графики и числа. Ответ на каждый вопрос — 2–4 предложения.

1. Во сколько раз различаются по масштабу признаки Spambase (см. таблицу после загрузки данных)? Почему
   без масштабирования SVM с RBF-ядром работает так плохо (задание 2.1)? Нужно ли было бы масштабирование
   дереву решений?
2. Сравните линейное и RBF-ядро на Spambase (задания 2.2–2.4) и на синтетических данных (задание 3.1). Почему
   на окружностях разница огромная, а на Spambase небольшая? Почему для окружностей хватает полиномиального
   ядра степени 2?
3. Как граница и число опорных векторов зависят от $\gamma$ при фиксированном $C$ (задание 3.2)? Что происходит
   при очень большом и очень малом $\gamma$? На какой метод из лекции 01 похож SVM с большим $\gamma$ и почему?
4. Почему число опорных векторов убывает с ростом $C$? Свяжите это с шириной полосы и с тем, какие объекты
   становятся опорными.
5. Опишите карты качества из задания 3.3: где модель недообучается, где переобучается? Почему область хорошего
   качества вытянута по диагонали — что общего у увеличения $C$ и увеличения $\gamma$?
6. Что дал сдвиг порога в задании 2.5: как изменились precision, recall и число потерянных писем? Почему порог
   подбирали по кросс-валидации на обучающей выборке, а не по тестовой? Почему для этого не нужны вероятности?
7. По заданию 1.3: какая доля объектов оказалась опорными векторами и сколько из них нарушители? Где на
   плоскости лежат объекты каждой из трёх групп? Почему решающая функция зависит только от опорных векторов?

*Ваш ответ:*